In [3]:
# 📦 Install dependencies (run once if needed)
# !pip install pandas scikit-learn nltk

# 📚 Imports
import pandas as pd
import nltk
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 🔽 Download NLTK data (run once)
nltk.download('stopwords')
nltk.download('wordnet')

# 📂 Load dataset
df = pd.read_csv('food_and_drink.csv')

# 👀 Preview data
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,name,description,category
0,Pizza,Cheesy Italian dish with tomato sauce,Food
1,Latte,Coffee drink with milk foam,Drink
2,Burger,Grilled beef patty with bun,Food
3,Smoothie,Blended fruit beverage,Drink


In [4]:
# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Text cleaning function
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation
    
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

# Apply preprocessing to description column
df['clean_text'] = df['description'].apply(preprocess)

df[['description', 'clean_text']].head()

,description,clean_text
0,Cheesy Italian dish with tomato sauce,cheesy italian dish tomato sauce
1,Coffee drink with milk foam,coffee drink milk foam
2,Grilled beef patty with bun,grilled beef patty bun
3,Blended fruit beverage,blended fruit beverage


In [5]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['clean_text'])

print("Feature shape:", X.shape)

Feature shape: (4, 16)


In [6]:
# Choose number of clusters (try 2–5 depending on dataset)
k = 3

kmeans = KMeans(n_clusters=k, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

df.head()

,name,description,category,clean_text,cluster
0,Pizza,Cheesy Italian dish with tomato sauce,Food,cheesy italian dish tomato sauce,2
1,Latte,Coffee drink with milk foam,Drink,coffee drink milk foam,0
2,Burger,Grilled beef patty with bun,Food,grilled beef patty bun,0
3,Smoothie,Blended fruit beverage,Drink,blended fruit beverage,1


In [7]:
# Group results
for i in range(k):
    print(f"\n--- Cluster {i} ---")
    print(df[df['cluster'] == i]['name'].values)


--- Cluster 0 ---
<ArrowStringArray>
['Latte', 'Burger']
Length: 2, dtype: str

--- Cluster 1 ---
<ArrowStringArray>
['Smoothie']
Length: 1, dtype: str

--- Cluster 2 ---
<ArrowStringArray>
['Pizza']
Length: 1, dtype: str


In [8]:
import numpy as np

terms = vectorizer.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    print(f"\nCluster {i} keywords:")
    for ind in order_centroids[i, :10]:
        print(terms[ind], end=", ")


Cluster 0 keywords:
patty, milk, grilled, foam, drink, coffee, bun, beef, tomato, sauce, 
Cluster 1 keywords:
fruit, blended, beverage, tomato, sauce, patty, milk, italian, grilled, foam, 
Cluster 2 keywords:
tomato, sauce, italian, dish, cheesy, patty, milk, grilled, fruit, foam, 